# Person Re-Identification: Experimental Demonstration and Analysis

## 1. Mục tiêu và phạm vi notebook

Notebook này được biên soạn theo định hướng học thuật nhằm **trình bày quy trình thực nghiệm** cho bài toán Person Re-Identification (ReID), đồng thời hỗ trợ trình bày video đồ án một cách hệ thống và có khả năng kiểm chứng.

Cụ thể, tài liệu này phục vụ các mục tiêu sau:

1. Minh họa bản chất bài toán Person ReID trong thiết lập query-gallery.
2. Đánh giá mô hình baseline TransReID và ba nhánh cải tiến được đề xuất.
3. Phân tích định lượng thông qua các chỉ số mAP, Rank-1, Rank-5 và Rank-10.
4. Phân tích định tính thông qua Top-K retrieval, reliability heatmap và semantic alignment.
5. Đảm bảo khả năng tái lập thực nghiệm thông qua kiểm tra checkpoint, cấu hình và các lệnh chạy tương ứng.

Phần huấn luyện được giữ ở dạng phụ lục tham khảo, trong khi trọng tâm trình bày đặt vào đánh giá từ checkpoint và diễn giải kết quả.

## 2. Bài toán Person Re-Identification

Person Re-Identification được phát biểu như một bài toán truy hồi hình ảnh có giám sát yếu trong môi trường đa camera. Với một ảnh **query** (ảnh người cần tìm), hệ thống phải truy hồi trong tập **gallery** (ảnh ứng viên thu từ các camera khác nhau) và sắp xếp các ứng viên theo mức độ tương đồng với query.

Về mặt biểu diễn, mỗi ảnh được ánh xạ thành một **embedding** trong không gian đặc trưng. Từ đó, một hàm khoảng cách hoặc độ tương đồng (distance/similarity) được sử dụng để tạo **ranking** các ảnh gallery đối với từng query. Do đó, ReID khác biệt căn bản so với classification: đầu ra không phải một nhãn lớp duy nhất, mà là một danh sách ứng viên có thứ hạng.

Bài toán này đặc biệt thách thức do đồng thời chịu tác động của nhiều yếu tố: sai khác miền giữa camera (cross-camera shift), biến thiên tư thế (pose variation), thay đổi điều kiện chiếu sáng (illumination), che khuất cục bộ (occlusion), nhiễu nền (background clutter), và hiện tượng nhiều người có ngoại hình tương tự nhau (visual ambiguity). Vì vậy, việc phân tích đồng thời kết quả định lượng và định tính là cần thiết để hiểu hành vi mô hình một cách đầy đủ.

## 3. Thiết lập thực nghiệm và tham số điều khiển

Các biến `RUN_*` quyết định notebook sẽ thực thi lệnh tính toán hay chỉ hiển thị cấu hình và artifact đã có sẵn.

- `RUN_EVAL = True`: thực thi `test.py` để đánh giá các checkpoint.
- `RUN_VISUALIZE = True`: chạy lại các script trực quan hóa kết quả.
- `RUN_TRAIN = True`: thực thi lệnh huấn luyện (không khuyến nghị trong phiên demo ngắn).
- `FAST_TRAIN_DEMO = True`: nếu huấn luyện được bật, chỉ chạy cấu hình rút gọn nhằm kiểm tra pipeline, không dùng làm kết quả báo cáo chính thức.

In [ ]:
# Cell này thiết lập toàn bộ import và biến đường dẫn dùng xuyên suốt notebook.
# Mục tiêu là tập trung cấu hình tại một nơi để giảm sai lệch khi tái lập thực nghiệm.
from pathlib import Path
import os
import sys
import csv
import json
import shlex
import random
import subprocess
from collections import defaultdict
from datetime import datetime

from IPython.display import display, Markdown, Image as IPyImage

PROJECT_ROOT = Path(r"D:\DAI_HOC\NAM_3\SEM 2\TGMT\PROJECT\K23-TGMT")
SOURCE_ROOT = PROJECT_ROOT / "1.3 Source code"
DEMO_ROOT = PROJECT_ROOT / "1.4 Demo"
ARTIFACT_ROOT = SOURCE_ROOT / "transreid_artifacts"

LOCAL_SRC = SOURCE_ROOT / "local-reliability"
SEMANTIC_SRC = SOURCE_ROOT / "semantic"
TOOLS_DIR = ARTIFACT_ROOT / "tools"

DATASET_ROOT = Path(r"D:\DAI_HOC\NAM_3\SEM 2\TGMT\PROJECT\Requirement 2 & 3\Colab\data\market1501")
DATA_PARENT = DATASET_ROOT.parent

DEMO_OUTPUT = DEMO_ROOT / "generated_outputs"
DEMO_OUTPUT.mkdir(parents=True, exist_ok=True)

# Cờ điều khiển: ưu tiên chạy evaluation/visualization thay vì huấn luyện trong phiên demo.
RUN_INSTALL = False
RUN_TRAIN = False
FAST_TRAIN_DEMO = True
RUN_EVAL = False
RUN_VISUALIZE = False

DEVICE_ID = "0"
PYTHON = sys.executable

print("Project:", PROJECT_ROOT)
print("Dataset exists:", DATASET_ROOT.exists(), DATASET_ROOT)
print("Artifacts:", ARTIFACT_ROOT)

## 4. Baseline và ba nhánh phương pháp đề xuất

Thực nghiệm sử dụng **TransReID** làm mô hình nền (baseline) để làm mốc so sánh nhất quán giữa các biến thể.

- **Baseline TransReID**: đại diện cho kiến trúc chuẩn, cung cấp đường cơ sở về năng lực truy hồi trên cùng tập dữ liệu và giao thức đánh giá.
- **Local Reliability**: bổ sung cơ chế ước lượng độ tin cậy cho patch/local token, nhằm giảm ảnh hưởng của các vùng không ổn định (che khuất, nền nhiễu, chi tiết không phân biệt).
- **Semantic Alignment**: đưa tín hiệu semantic/body-part vào quá trình học patch token để tăng tính cấu trúc của biểu diễn và cải thiện nhất quán ngữ nghĩa liên camera.
- **Semantic + Reliability**: kết hợp đồng thời hai hướng trên, kỳ vọng khai thác bổ sung lẫn nhau giữa thông tin ngữ nghĩa và trọng số tin cậy cục bộ.

Cách thiết kế này cho phép phân tích đóng góp của từng thành phần một cách rõ ràng thông qua so sánh định lượng và định tính.

In [ ]:
# Cell này khai báo baseline và các nhánh đề xuất dưới dạng dictionary thống nhất.
# Cấu trúc BRANCHES giúp tránh hard-code lặp lại ở các bước evaluate/train/visualize.
BRANCHES = {
    "baseline": {
        "title": "Baseline TransReID",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_fair.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_baseline" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_baseline",
        "note": "Mô hình nền TransReID.",
    },
    "local": {
        "title": "Local Reliability",
        "source": LOCAL_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_local_reliability.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_local_reliability" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_local_reliability",
        "note": "Ước lượng reliability score cho patch/local token.",
    },
    "semantic": {
        "title": "Semantic Alignment",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_sem_align.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_sem_align" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_semantic_alignment",
        "note": "Dùng semantic mask để giám sát patch token.",
    },
    "combine": {
        "title": "Semantic + Reliability",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_sem_align_reliability.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_sem_align_reliability" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_semantic_reliability",
        "note": "Kết hợp semantic supervision và reliability-aware local fusion.",
    },
}

rows = []
for key, branch in BRANCHES.items():
    ckpt = branch["checkpoint"]
    cfg = branch["config"]
    rows.append({
        "key": key,
        "model": branch["title"],
        "source": str(branch["source"]),
        "config_exists": cfg.exists(),
        "checkpoint_exists": ckpt.exists(),
        "checkpoint_size_MB": round(ckpt.stat().st_size / 1024**2, 2) if ckpt.exists() else None,
        "note": branch["note"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    # Fallback in text mode để notebook vẫn hoạt động khi thiếu pandas.
    for row in rows:
        print(row)

## 5. Kiểm tra điều kiện thực nghiệm

Mục này thực hiện kiểm tra tính sẵn sàng của toàn bộ pipeline trước khi đánh giá:

- Đường dẫn dự án, mã nguồn, artifact và thư mục công cụ.
- Cấu trúc dữ liệu ReID và số lượng ảnh theo từng split.
- Cấu hình và checkpoint của bốn nhánh mô hình.
- Các script trực quan hóa quan trọng dùng cho phân tích định tính.

Ký hiệu trạng thái:

- `[OK]`: thành phần sẵn sàng cho thực nghiệm.
- `[WARNING]`: thành phần phụ còn thiếu, có thể ảnh hưởng một phần phân tích.
- `[MISSING]`: thiếu thành phần thiết yếu, cần khắc phục trước khi chạy các bước chính.

In [ ]:
# Cell này thực hiện kiểm tra điều kiện thực nghiệm trước khi chạy đánh giá/visualization.
# Thiết kế theo hướng fail-fast cho thành phần bắt buộc, nhưng vẫn cho phép tiếp tục với cảnh báo ở thành phần phụ.
def count_images(split_name):
    """Đếm số ảnh .jpg trong một split của dataset.

    Args:
        split_name: Tên thư mục split (ví dụ: 'query', 'bounding_box_test').

    Returns:
        Số lượng ảnh dạng .jpg. Trả về 0 nếu split không tồn tại.
    """
    split_dir = DATASET_ROOT / split_name
    if not split_dir.exists():
        return 0
    return len(list(split_dir.glob("*.jpg")))


def preflight_check(verbose=True, strict=True):
    """Kiểm tra trạng thái sẵn sàng của pipeline thực nghiệm.

    Hàm này tổng hợp kiểm tra cho đường dẫn dự án, dữ liệu, checkpoint/config
    của các nhánh mô hình và các script trực quan hóa thiết yếu.

    Args:
        verbose: Nếu True, in báo cáo chi tiết ra output cell.
        strict: Nếu True, raise RuntimeError khi thiếu thành phần bắt buộc.

    Returns:
        Dict gồm danh sách log và các mục thiếu bắt buộc.

    Raises:
        RuntimeError: Khi strict=True và thiếu thành phần bắt buộc cho demo checkpoint.
    """
    logs = []
    hard_missing = []

    def push(status, label, path=None, required=False, detail=""):
        msg = f"[{status}] {label}"
        if path is not None:
            msg += f" -> {path}"
        if detail:
            msg += f" ({detail})"
        logs.append(msg)
        if status == "MISSING" and required:
            hard_missing.append(msg)

    # 1) Kiểm tra đường dẫn lõi của dự án và artifact.
    core_paths = [
        ("PROJECT_ROOT", PROJECT_ROOT, True),
        ("SOURCE_ROOT", SOURCE_ROOT, True),
        ("ARTIFACT_ROOT", ARTIFACT_ROOT, True),
        ("LOCAL_SRC", LOCAL_SRC, True),
        ("SEMANTIC_SRC", SEMANTIC_SRC, True),
        ("TOOLS_DIR", TOOLS_DIR, False),
    ]
    for name, p, required in core_paths:
        push("OK" if p.exists() else "MISSING", name, p, required=required)

    # 2) Kiểm tra dataset và các split dùng trong evaluation.
    push("OK" if DATASET_ROOT.exists() else "MISSING", "DATASET_ROOT", DATASET_ROOT, required=True)
    for split in ["bounding_box_train", "query", "bounding_box_test"]:
        split_dir = DATASET_ROOT / split
        if split_dir.exists():
            push("OK", f"Split {split}", split_dir, detail=f"{count_images(split)} images")
        else:
            required = split in {"query", "bounding_box_test"}
            push("MISSING", f"Split {split}", split_dir, required=required)

    # 3) Kiểm tra config và checkpoint theo từng nhánh mô hình.
    for key, branch in BRANCHES.items():
        cfg = branch["config"]
        ckpt = branch["checkpoint"]
        push("OK" if cfg.exists() else "MISSING", f"{key}.config", cfg, required=True)
        if ckpt.exists():
            mb = round(ckpt.stat().st_size / 1024**2, 2)
            push("OK", f"{key}.checkpoint", ckpt, required=True, detail=f"{mb} MB")
        else:
            push("MISSING", f"{key}.checkpoint", ckpt, required=True)

    # 4) Kiểm tra sự tồn tại của các script visualization/evaluation quan trọng.
    script_checks = [
        ("eval.local.test", LOCAL_SRC / "test.py", True),
        ("eval.semantic.test", SEMANTIC_SRC / "test.py", True),
        ("viz.report", TOOLS_DIR / "generate_report_visuals.py", False),
        ("viz.multimodel_topk", TOOLS_DIR / "visualize_multimodel_retrieval.py", False),
        ("viz.local_suite", LOCAL_SRC / "visualize_person_reid_suite.py", False),
        ("viz.semantic_basic", SEMANTIC_SRC / "visualize_semantic_no_checkpoint.py", False),
        ("viz.semantic_real", SEMANTIC_SRC / "visualize_real_semantic_examples.py", False),
        ("viz.semantic_extra", SEMANTIC_SRC / "visualize_semantic_extra.py", False),
    ]
    for label, p, required in script_checks:
        status = "OK" if p.exists() else ("MISSING" if required else "WARNING")
        push(status, label, p, required=required)

    if verbose:
        print("Python:", sys.version.split()[0])
        try:
            import torch
            print("Torch:", torch.__version__)
            print("CUDA available:", torch.cuda.is_available())
            if torch.cuda.is_available():
                print("GPU:", torch.cuda.get_device_name(0))
        except Exception as exc:
            # Không chặn notebook nếu chỉ thiếu thông tin torch/cuda.
            print("[WARNING] Torch check failed:", exc)

        print("\n--- Preflight Summary ---")
        for line in logs:
            print(line)

    if hard_missing and strict:
        raise RuntimeError(
            "Thiếu thành phần bắt buộc cho demo checkpoint. Hãy sửa các mục [MISSING] sau:\n"
            + "\n".join(hard_missing)
        )

    return {"logs": logs, "hard_missing": hard_missing}


_ = preflight_check(verbose=True, strict=True)

### Hàm tiện ích nội bộ

Các hàm dưới đây hỗ trợ thực thi lệnh, chuẩn hóa hiển thị artifact và giảm trùng lặp mã nguồn.
Việc tách thành hàm riêng giúp notebook rõ cấu trúc và thuận lợi cho tái lập thực nghiệm.

In [ ]:
# Cell tiện ích dùng chung cho toàn notebook: format command, chạy subprocess, hiển thị ảnh và CSV.
# Các hàm được thiết kế để output gọn và thân thiện cho môi trường trình bày.
def quote_cmd(cmd):
    """Chuẩn hóa danh sách tham số lệnh thành chuỗi dễ đọc.

    Args:
        cmd: Danh sách thành phần lệnh (str/Path/number).

    Returns:
        Chuỗi command-line đã được quote an toàn để hiển thị.
    """
    return " ".join(shlex.quote(str(x)) for x in cmd)


def run_command(cmd, cwd, run=False, env=None):
    """Thực thi một lệnh shell bằng subprocess với chế độ bật/tắt.

    Args:
        cmd: Danh sách tham số lệnh.
        cwd: Thư mục làm việc khi chạy lệnh.
        run: Nếu False, chỉ in lệnh để tham khảo; nếu True thì thực thi.
        env: Dict biến môi trường bổ sung, sẽ merge với os.environ.

    Returns:
        CompletedProcess khi lệnh chạy thành công; None nếu run=False.

    Raises:
        RuntimeError: Khi lệnh trả về return code khác 0.
    """
    print("CWD:", cwd)
    print("CMD:", quote_cmd(cmd))
    if not run:
        print("Skip: đặt cờ RUN_* = True để chạy lệnh này.")
        return None
    env_all = os.environ.copy()
    if env:
        env_all.update(env)
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        env=env_all,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    # Chỉ in đoạn cuối để tránh output quá dài khi quay video hoặc review notebook.
    print(proc.stdout[-6000:])
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with return code {proc.returncode}")
    return proc


def show_image(path, width=1000):
    """Hiển thị ảnh nếu tồn tại, kèm thông báo rõ khi thiếu artifact.

    Args:
        path: Đường dẫn tới ảnh cần hiển thị.
        width: Chiều rộng hiển thị (pixel) trong notebook.

    Returns:
        None. Hàm dùng cho side-effect hiển thị.
    """
    path = Path(path)
    if not path.exists():
        display(Markdown(f"**Chưa có ảnh:** `{path}`"))
        return
    display(Markdown(f"`{path}`"))
    display(IPyImage(filename=str(path), width=width))


def show_csv(path):
    """Đọc và hiển thị tệp CSV theo định dạng bảng.

    Args:
        path: Đường dẫn tệp CSV.

    Returns:
        None. Nếu thiếu pandas, fallback sang in text thuần.
    """
    path = Path(path)
    if not path.exists():
        print("Missing:", path)
        return
    try:
        import pandas as pd
        display(pd.read_csv(path))
    except Exception:
        print(path.read_text(encoding="utf-8-sig"))

## 6. Khảo sát dữ liệu và minh họa cùng danh tính qua nhiều camera

Mục này trực quan hóa tính chất cốt lõi của ReID:

1. Cùng một danh tính (`pid`) có thể xuất hiện rất khác nhau dưới các camera khác nhau.
2. Nhiều danh tính khác có thể mang ngoại hình tương tự, tạo nhiễu đối với truy hồi.

Notebook sẽ parse `pid`/`camid` từ tên tệp ảnh theo chuẩn thường dùng của Market-1501/Duke, sau đó hiển thị:

- Một nhóm ảnh cùng `pid` (ưu tiên đa camera).
- Một nhóm distractor khác `pid` để minh họa độ mơ hồ thị giác.

In [ ]:
# Cell này xây dựng các hàm hỗ trợ khảo sát dữ liệu và minh họa same-identity cross-camera.
# Mục tiêu là cung cấp bằng chứng trực quan về độ khó của bài toán ReID trước khi vào đánh giá mô hình.
def parse_pid_camid(img_path):
    """Trích xuất pid và camid từ tên tệp ảnh theo chuẩn Market-1501/Duke.

    Args:
        img_path: Đường dẫn ảnh hoặc tên tệp ảnh.

    Returns:
        Tuple (pid, camid). Nếu không parse được camid, trả về 'unknown'.
    """
    stem = Path(img_path).stem
    parts = stem.split("_")
    pid = parts[0] if parts else "unknown"
    camid = "unknown"
    for part in parts[1:]:
        if part.startswith("c") and len(part) > 1 and part[1].isdigit():
            camid = part[1:]
            break
    return pid, camid


def list_split_images(split_name, limit=None):
    """Liệt kê ảnh trong một split dataset theo thứ tự ổn định.

    Args:
        split_name: Tên split (ví dụ: 'query', 'bounding_box_test').
        limit: Giới hạn số ảnh trả về; None nghĩa là lấy toàn bộ.

    Returns:
        Danh sách Path đến tệp .jpg. Trả về list rỗng nếu split không tồn tại.
    """
    split_dir = DATASET_ROOT / split_name
    if not split_dir.exists():
        return []
    images = sorted(split_dir.glob("*.jpg"))
    if limit is not None:
        return images[:limit]
    return images


def build_pid_index(images):
    """Tạo chỉ mục pid -> danh sách (image_path, camid).

    Args:
        images: Danh sách đường dẫn ảnh.

    Returns:
        defaultdict(list) ánh xạ pid sang các mẫu ảnh tương ứng.

    Notes:
        Bỏ qua pid '-1', '0000' và giá trị không hợp lệ để giảm nhiễu.
    """
    index = defaultdict(list)
    for p in images:
        pid, camid = parse_pid_camid(p)
        if pid in {"-1", "0000", "unknown"}:
            continue
        index[pid].append((p, camid))
    return index


def show_same_id_cross_camera(max_same_id=6, max_distractor=6):
    """Hiển thị cùng danh tính qua nhiều camera và một hàng distractor.

    Args:
        max_same_id: Số ảnh tối đa của cùng pid ở hàng trên.
        max_distractor: Số ảnh tối đa khác pid ở hàng dưới.

    Returns:
        None. Hàm hiển thị trực tiếp bằng matplotlib.

    Error Handling:
        - In cảnh báo nếu thiếu matplotlib/Pillow.
        - In cảnh báo rõ khi thiếu dữ liệu đủ điều kiện.
    """
    try:
        import matplotlib.pyplot as plt
        from PIL import Image
    except Exception as exc:
        print("[WARNING] Missing matplotlib/Pillow for preview:", exc)
        return

    gallery_images = list_split_images("bounding_box_test")
    if not gallery_images:
        print("[MISSING] Không tìm thấy ảnh trong split bounding_box_test.")
        return

    pid_index = build_pid_index(gallery_images)
    candidates = []
    for pid, items in pid_index.items():
        cams = {cam for _, cam in items}
        if len(items) >= 4:
            candidates.append((pid, len(cams), len(items), items))

    if not candidates:
        print("[WARNING] Không đủ identity có >=4 ảnh để minh họa cross-camera.")
        return

    # Ưu tiên identity có nhiều camera nhất, sau đó nhiều ảnh nhất.
    candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
    chosen_pid, num_cams, _, items = candidates[0]

    same_samples = sorted(items, key=lambda x: x[1])[:max_same_id]

    other_pids = [pid for pid in pid_index.keys() if pid != chosen_pid]
    distractors = []
    for pid in other_pids:
        if len(distractors) >= max_distractor:
            break
        distractors.append((pid_index[pid][0][0], parse_pid_camid(pid_index[pid][0][0])[1], pid))

    n_cols = max(len(same_samples), len(distractors), 4)
    fig, axes = plt.subplots(2, n_cols, figsize=(3.2 * n_cols, 8))

    for c in range(n_cols):
        axes[0, c].axis("off")
        axes[1, c].axis("off")

    for c, (img_path, camid) in enumerate(same_samples):
        axes[0, c].imshow(Image.open(img_path).convert("RGB"))
        axes[0, c].set_title(f"same pid={chosen_pid}\ncam={camid}", fontsize=10)

    for c, (img_path, camid, pid) in enumerate(distractors):
        axes[1, c].imshow(Image.open(img_path).convert("RGB"))
        axes[1, c].set_title(f"distractor pid={pid}\ncam={camid}", fontsize=10)

    fig.suptitle(
        f"Same identity across cameras (pid={chosen_pid}, cams={num_cams}) vs distractors",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()


show_same_id_cross_camera(max_same_id=6, max_distractor=6)

## 7. Thiết lập query-gallery

Phần này minh họa cấu hình truy hồi chuẩn:

- Một ảnh query đóng vai trò mẫu cần tìm.
- Tập gallery gồm các ứng viên có thể chứa cả true match và distractor.

Kết quả của mô hình sẽ là danh sách gallery được xếp hạng theo mức độ tương đồng embedding với query. Đây là cơ sở để diễn giải các chỉ số Rank-k và phân tích Top-K retrieval ở các mục sau.

In [ ]:
# Cell này minh họa setup query-gallery với một ví dụ cụ thể (1 query + nhiều candidate).
# Việc quan sát mẫu này giúp người đọc hiểu trực tiếp cơ chế ranking trước khi xem Top-K của mô hình.
def show_query_gallery_example(num_gallery=5):
    """Hiển thị một ví dụ query-gallery gồm positive và distractor.

    Args:
        num_gallery: Số lượng ảnh gallery cần hiển thị (bao gồm 1 positive nếu tìm được).

    Returns:
        None. Hàm hiển thị bằng matplotlib.

    Error Handling:
        - Cảnh báo nếu thiếu thư viện hiển thị.
        - Cảnh báo rõ khi không có dữ liệu query/gallery hợp lệ.
    """
    try:
        import matplotlib.pyplot as plt
        from PIL import Image
    except Exception as exc:
        print("[WARNING] Missing matplotlib/Pillow for query-gallery preview:", exc)
        return

    query_images = list_split_images("query")
    gallery_images = list_split_images("bounding_box_test")

    if not query_images or not gallery_images:
        print("[MISSING] Cần có cả split query và bounding_box_test để minh họa query-gallery.")
        return

    gallery_index = build_pid_index(gallery_images)

    selected_query = None
    selected_positive = None
    for q in query_images:
        q_pid, q_cam = parse_pid_camid(q)
        if q_pid in {"-1", "0000", "unknown"}:
            continue
        positives = [item for item in gallery_index.get(q_pid, []) if item[0].name != q.name]
        if positives:
            selected_query = q
            selected_positive = positives[0][0]
            break

    if selected_query is None:
        print("[WARNING] Không tìm thấy query có positive match trong gallery.")
        return

    q_pid, q_cam = parse_pid_camid(selected_query)

    negatives = []
    for g in gallery_images:
        g_pid, g_cam = parse_pid_camid(g)
        if g_pid != q_pid and g_pid not in {"-1", "0000", "unknown"}:
            negatives.append(g)
        if len(negatives) >= max(1, num_gallery - 1):
            break

    gallery_show = [selected_positive] + negatives

    n_cols = 1 + len(gallery_show)
    fig, axes = plt.subplots(1, n_cols, figsize=(4.0 * n_cols, 5.2))

    axes[0].imshow(Image.open(selected_query).convert("RGB"))
    axes[0].set_title(f"QUERY\npid={q_pid} cam={q_cam}", fontsize=11)
    axes[0].axis("off")

    for i, g in enumerate(gallery_show, start=1):
        g_pid, g_cam = parse_pid_camid(g)
        is_match = g_pid == q_pid
        axes[i].imshow(Image.open(g).convert("RGB"))
        axes[i].set_title(
            f"GALLERY-{i}\npid={g_pid} cam={g_cam}\n{'MATCH' if is_match else 'DISTRACTOR'}",
            fontsize=10,
        )
        axes[i].axis("off")

    fig.suptitle("Query-Gallery setup for Person Re-ID", fontsize=14)
    plt.tight_layout()
    plt.show()


show_query_gallery_example(num_gallery=5)

## 8. Đánh giá từ checkpoint đã huấn luyện

Đây là bước đánh giá trung tâm của notebook. Với mỗi nhánh mô hình, `test.py` được gọi cùng cấu hình tương ứng để tính toán các chỉ số truy hồi tiêu chuẩn.

Khi đặt `RUN_EVAL = True`, kết quả đánh giá sẽ được ghi vào thư mục `generated_outputs/eval_*`, làm đầu vào cho các bước tổng hợp bảng và trực quan hóa tiếp theo.

In [ ]:
# Cell này xây dựng và thực thi lệnh evaluation cho từng nhánh từ checkpoint có sẵn.
# Mục tiêu là tạo kết quả định lượng đồng nhất để so sánh công bằng giữa các phương pháp.
def build_eval_command(key):
    """Tạo command đánh giá cho một nhánh mô hình.

    Args:
        key: Tên nhánh trong BRANCHES (baseline/local/semantic/combine).

    Returns:
        Danh sách tham số lệnh dùng cho subprocess.
    """
    branch = BRANCHES[key]
    cmd = [
        PYTHON, "test.py",
        "--config_file", branch["config"],
        "DATASETS.ROOT_DIR", str(DATA_PARENT),
        "MODEL.DEVICE_ID", DEVICE_ID,
        "TEST.WEIGHT", str(branch["checkpoint"]),
        "TEST.IMS_PER_BATCH", "128",
        "DATALOADER.NUM_WORKERS", "0",
        "OUTPUT_DIR", str(branch["output"]),
    ]
    return cmd

for key in ["baseline", "local", "semantic", "combine"]:
    display(Markdown(f"### Evaluate: {BRANCHES[key]['title']}"))
    cmd = build_eval_command(key)
    run_command(cmd, cwd=BRANCHES[key]["source"], run=RUN_EVAL)

## 9. Bảng chỉ số tổng hợp và cách diễn giải

Cell kế tiếp đọc hai artifact:

- `metrics_summary.csv`: bảng tổng hợp các chỉ số truy hồi (mAP, Rank-1, Rank-5, Rank-10) giữa các nhánh.
- `latency_summary.csv`: bảng tổng hợp độ trễ suy luận phục vụ phân tích đánh đổi chính xác-hiệu năng.

Hai tệp này thường được tổng hợp từ log đánh giá. Nếu vừa chạy lại `test.py`, cần bảo đảm bước tổng hợp log sang CSV đã hoàn tất trước khi đọc bảng.

**Cách đọc nhanh các chỉ số:**

- **mAP** phản ánh chất lượng ranking tổng thể trên toàn bộ query.
- **Rank-1** phản ánh khả năng đưa đúng danh tính lên vị trí đầu.
- **Rank-5/Rank-10** phản ánh độ ổn định khi xét nhiều vị trí đầu trong danh sách truy hồi.

In [ ]:
# Cell này đọc các bảng tổng hợp metric/latency từ artifact đã chuẩn bị trước.
# Nếu tệp chưa tồn tại, show_csv sẽ in thông báo rõ để tránh nhầm lẫn khi thuyết trình.
show_csv(ARTIFACT_ROOT / "metrics_summary.csv")
show_csv(ARTIFACT_ROOT / "latency_summary.csv")

## 10. Phân tích định lượng qua trực quan hóa

Script `generate_report_visuals.py` tạo các biểu đồ phục vụ phân tích định lượng gồm:

- So sánh mAP và CMC Rank-k giữa các mô hình.
- Mức thay đổi so với baseline.
- Diễn biến đường học/đánh giá theo epoch (nếu có log tương ứng).
- Quan hệ đánh đổi giữa độ chính xác và độ trễ.

Các biểu đồ này giúp đối chiếu tính nhất quán giữa chỉ số bảng và xu hướng tổng thể của mô hình.

In [ ]:
# Cell này tạo/cập nhật các biểu đồ định lượng tổng hợp từ artifact đã có.
# Biểu đồ được hiển thị với kích thước lớn để thuận tiện cho phân tích khi trình bày video.
cmd = [PYTHON, str(TOOLS_DIR / "generate_report_visuals.py")]
run_command(cmd, cwd=ARTIFACT_ROOT, run=RUN_VISUALIZE)

quant_dir = ARTIFACT_ROOT / "visualizations" / "quantitative"
show_image(quant_dir / "09_summary_table.png", width=1000)
show_image(quant_dir / "01_metrics_grouped_bar.png", width=1000)
show_image(quant_dir / "02_delta_vs_baseline.png", width=900)
show_image(quant_dir / "06_validation_map_rank1_curves.png", width=1000)
show_image(quant_dir / "03_accuracy_latency_tradeoff.png", width=900)

## 11. Phân tích định tính thông qua Top-K retrieval

Top-K retrieval cho phép quan sát trực tiếp hành vi xếp hạng của mô hình trên cùng một query.

**Nguyên tắc diễn giải:**

1. Kiểm tra vị trí xuất hiện của true match trong danh sách đầu.
2. So sánh mức ổn định giữa các nhánh khi điều kiện ảnh khó (góc nhìn, che khuất, nền nhiễu).
3. Ghi nhận các trường hợp nhầm lẫn có tính hệ thống để liên hệ với phần failure analysis.

Phần này đặc biệt quan trọng vì phản ánh trực tiếp bản chất ranking của bài toán ReID.

In [ ]:
# Cell này tạo hình so sánh Top-K retrieval giữa 4 mô hình trên cùng tập truy vấn.
# Kết quả giúp đối chiếu trực tiếp thứ hạng true match giữa các nhánh phương pháp.
cmd = [
    PYTHON, str(TOOLS_DIR / "visualize_multimodel_retrieval.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(ARTIFACT_ROOT / "visualizations" / "qualitative"),
    "--topk", "5",
    "--num-queries", "3",
    "--num-distractors", "60",
    "--batch-size", "32",
]
run_command(cmd, cwd=ARTIFACT_ROOT, run=RUN_VISUALIZE)

qual_dir = ARTIFACT_ROOT / "visualizations" / "qualitative"
show_image(qual_dir / "01_same_query_topk_comparison.png", width=950)
show_image(qual_dir / "02_true_match_rank_matrix.png", width=1000)

## 12. Trực quan hóa Local Reliability

Mục tiêu của reliability heatmap là cho thấy mô hình đang phân bổ mức tin cậy không gian như thế nào trên ảnh người.

**Cách đọc reliability heatmap:**

- Vùng có độ tin cậy cao thường tương ứng chi tiết nhận dạng ổn định.
- Vùng có độ tin cậy thấp thường gắn với nền, che khuất hoặc nhiễu thị giác.
- So sánh heatmap với kết quả retrieval giúp lý giải vì sao một truy hồi thành công hoặc thất bại.

Những minh họa này hỗ trợ diễn giải cơ chế hoạt động của nhánh Local Reliability ngoài các con số định lượng.

In [ ]:
# Cell này chạy bộ trực quan hóa chuyên biệt cho nhánh Local Reliability.
# Mục tiêu là quan sát đồng thời retrieval, heatmap độ tin cậy và các trường hợp thất bại.
local_out = ARTIFACT_ROOT / "visualizations" / "local_reliability"
cmd = [
    PYTHON, str(LOCAL_SRC / "visualize_person_reid_suite.py"),
    "--checkpoint", str(BRANCHES["local"]["checkpoint"]),
    "--config-file", str(BRANCHES["local"]["config"]),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(local_out),
    "--num-queries", "3",
    "--topk", "5",
    "--batch-size", "16",
]
run_command(cmd, cwd=LOCAL_SRC, run=RUN_VISUALIZE)

show_image(local_out / "01_ranked_topk_results.png", width=1000)
show_image(local_out / "03_multi_query_retrieval_grid.png", width=1000)
show_image(local_out / "05_failure_cases.png", width=850)
show_image(local_out / "07_reliability_heatmap.png", width=900)
show_image(local_out / "08_synthetic_occlusion_target.png", width=950)
show_image(local_out / "06_embedding_pca.png", width=800)

## 13. Trực quan hóa Semantic Alignment

Semantic Alignment nhằm đưa ràng buộc ngữ nghĩa theo vùng cơ thể vào biểu diễn patch token.

**Cách đọc semantic visualization:**

- Đối chiếu giữa mask/ngữ nghĩa và vị trí patch để đánh giá mức phù hợp hình học-ngữ nghĩa.
- Quan sát mức nhất quán semantic của cùng danh tính qua các camera khác nhau.
- Phân tích các vùng semantic dễ gây nhầm lẫn khi người có trang phục tương tự.

Các hình này giúp kiểm tra giả thuyết rằng tín hiệu semantic cải thiện tính cấu trúc của embedding.

In [ ]:
# Cell này tạo trực quan hóa cho nhánh Semantic Alignment.
# Các hình đầu ra hỗ trợ phân tích mức phù hợp giữa semantic mask và biểu diễn patch token.
sem_out = ARTIFACT_ROOT / "visualizations" / "semantic"

cmd_basic = [
    PYTHON, str(SEMANTIC_SRC / "visualize_semantic_no_checkpoint.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
    "--num-samples", "3",
]
run_command(cmd_basic, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)

# Hai script dưới đây dùng human parser thật. Nếu máy chưa cache model, lần đầu có thể cần tải model.
cmd_real = [
    PYTHON, str(SEMANTIC_SRC / "visualize_real_semantic_examples.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
    "--num-images", "2",
]
cmd_extra = [
    PYTHON, str(SEMANTIC_SRC / "visualize_semantic_extra.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
]

run_command(cmd_real, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)
run_command(cmd_extra, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)

show_image(sem_out / "01_real_semantic_examples.png", width=1000)
show_image(sem_out / "02_real_vs_heuristic_mask.png", width=1000)
show_image(sem_out / "03_local_group_patch_targets.png", width=900)
show_image(sem_out / "04_semantic_patch_distribution.png", width=1000)
show_image(sem_out / "05_same_identity_semantic_consistency.png", width=980)
show_image(sem_out / "06_semantic_pipeline_diagram.png", width=1000)
show_image(sem_out / "07_semantic_overlay_grid.png", width=1000)

## 14. Phân tích failure case

Một báo cáo thực nghiệm đầy đủ cần trình bày cả trường hợp thành công và thất bại.

**Khung phân tích failure case đề xuất:**

1. Sai khác hình học lớn giữa query và gallery (pose/viewpoint mismatch).
2. Che khuất cục bộ làm mất thông tin nhận dạng quan trọng.
3. Nhiễu do trang phục tương tự hoặc nền phức tạp.
4. Mối liên hệ giữa vùng tin cậy thấp (reliability) và vị trí nhầm lẫn trong Top-K.

Phần này giúp xác định giới hạn hiện tại của phương pháp và định hướng cải tiến tiếp theo.

In [ ]:
# Cell này hiển thị nhanh nhóm hình liên quan failure case để phục vụ phân tích giới hạn mô hình.
# Nếu artifact chưa được sinh, notebook sẽ in thông báo rõ thay vì dừng đột ngột.
failure_candidates = [
    ARTIFACT_ROOT / "visualizations" / "local_reliability" / "05_failure_cases.png",
    ARTIFACT_ROOT / "visualizations" / "qualitative" / "01_same_query_topk_comparison.png",
    ARTIFACT_ROOT / "visualizations" / "qualitative" / "02_true_match_rank_matrix.png",
]

shown = 0
for img in failure_candidates:
    if img.exists():
        show_image(img, width=1100)
        shown += 1

if shown == 0:
    print("Chưa có artifact failure/qualitative. Hãy bật RUN_VISUALIZE = True để tạo lại hình.")

## 15. Phụ lục: lệnh huấn luyện và khả năng tái lập

Phần này cung cấp lệnh huấn luyện tương ứng cho từng nhánh nhằm phục vụ tái lập toàn bộ quy trình.

Trong bối cảnh quay video thuyết trình ngắn, không khuyến nghị chạy huấn luyện đầy đủ do chi phí thời gian lớn. Thay vào đó, nên sử dụng checkpoint đã huấn luyện sẵn để tập trung vào đánh giá và phân tích kết quả.

In [ ]:
# Cell này cung cấp lệnh huấn luyện tham khảo cho mục đích tái lập thực nghiệm.
# Trong phiên demo ngắn, nên giữ RUN_TRAIN=False để tránh thời gian chạy kéo dài.
def build_train_command(key, fast_demo=True):
    """Tạo command huấn luyện cho một nhánh mô hình.

    Args:
        key: Tên nhánh trong BRANCHES.
        fast_demo: Nếu True, thêm override rút gọn (1 epoch) để kiểm tra pipeline.

    Returns:
        Danh sách tham số lệnh huấn luyện cho subprocess.
    """
    branch = BRANCHES[key]
    output_dir = DEMO_OUTPUT / f"train_{key}"
    cmd = [
        PYTHON, "train.py",
        "--config_file", branch["config"],
        "DATASETS.ROOT_DIR", str(DATA_PARENT),
        "MODEL.DEVICE_ID", DEVICE_ID,
        "OUTPUT_DIR", str(output_dir),
    ]
    if fast_demo:
        cmd += [
            "MODEL.PRETRAIN_CHOICE", "none",
            "SOLVER.MAX_EPOCHS", "1",
            "SOLVER.EVAL_PERIOD", "1",
            "SOLVER.CHECKPOINT_PERIOD", "1",
            "SOLVER.IMS_PER_BATCH", "8",
            "TEST.IMS_PER_BATCH", "64",
            "DATALOADER.NUM_WORKERS", "0",
        ]
    return cmd

for key in ["baseline", "local", "semantic", "combine"]:
    display(Markdown(f"### Train reference: {BRANCHES[key]['title']}"))
    cmd = build_train_command(key, fast_demo=FAST_TRAIN_DEMO)
    run_command(cmd, cwd=BRANCHES[key]["source"], run=RUN_TRAIN)

## 16. Tóm tắt luận điểm trình bày

Gợi ý trình tự trình bày trong video theo phong cách học thuật:

1. Nêu phát biểu bài toán Person ReID và đặc trưng query-gallery ranking.
2. Trình bày baseline TransReID và động cơ của ba nhánh cải tiến.
3. Chứng minh điều kiện thực nghiệm hợp lệ thông qua preflight check.
4. Thực hiện đánh giá từ checkpoint để thu các chỉ số chuẩn.
5. Phân tích định lượng qua mAP/Rank-k và các biểu đồ tổng hợp.
6. Phân tích định tính qua Top-K retrieval, reliability heatmap và semantic visualization.
7. Thảo luận failure cases để làm rõ giới hạn của phương pháp.
8. Kết luận đóng góp, mức cải thiện quan sát được và hướng phát triển tiếp theo.